In [ ]:
import os
import pandas as pd
from groq import Groq
from datetime import datetime
import time
import numpy as np
from dotenv import load_dotenv

load_dotenv()

In [ ]:
russia_bc = ["Our special military operation aims to liberate Ukraine from neo-Nazi influence and reduce its military threat to Russia and its people. #Denazification #Demilitarization",
             "NATO's relentless eastward expansion threatens Russia's security. We must act to protect our borders and maintain strategic balance in Europe. #StopNATO",
             "The Ukrainian government has oppressed ethnic Russians in Donbas for years. Russia has a duty to protect these people and ensure their rights and safety. #ProtectingRussians"]

# The thirteen rhetorical techniques, in the form supplied to the generating
# model: "<name>: <definition>". create_response() recovers the name by
# splitting the value on the first colon, so the prefix must match the name
# exactly. Definitions are compiled from prior research that identifies and
# classifies the persuasive devices at work in political communication online
# (Chernyavskiy et al. 2024; Da San Martino et al. 2019; Salman et al. 2023),
# and are reproduced in Appendix A of the thesis.
rhetorical_techniques = {
    "Name-calling and labeling" : "Name-calling and labeling: Attacking individuals or groups with derogatory names or labels to discredit them without providing evidence.",
    "Glittering generalities" : "Glittering generalities: Using vague, emotionally appealing phrases to associate a person, group, or idea with positive attributes without providing specifics.",
    "Transfer" : "Transfer: Associating a person, group, or idea with something that is widely respected or reviled to influence perception.",
    "Testimonial" : "Testimonial: Using endorsements from respected individuals or authorities to lend credibility to a message, even if the endorsement is not directly relevant.",
    "Plain folks" : "Plain folks: Presenting a person or group as relatable, ordinary, or 'just like you' to gain trust and support.",
    "Card stacking" : "Card stacking: Selectively presenting information, often by omitting or minimizing facts that do not support the desired narrative.",
    "Bandwagon" : "Bandwagon: Appealing to the desire to belong or conform by suggesting that 'everyone' supports a particular idea or action.",
    "Fear mongering" : "Fear mongering: Using fear, anxiety, or threats to influence people's opinions or actions.",
    "Scapegoating" : "Scapegoating: Blaming a person, group, or issue for problems or difficulties, often without evidence or justification.",
    "False dichotomy" : "False dichotomy: Presenting a situation as having only two possible options, even when there may be more nuanced alternatives.",
    "Loaded language" : "Loaded language: Using emotionally charged words or phrases to evoke a strong response and influence opinion.",
    "Repetition" : "Repetition: Constantly repeating a message or idea to increase its perceived importance and acceptance.",
    "False Equivalence" : "False Equivalence: Drawing a comparison between two different actions, events, or situations that are not truly comparable in scale, severity, or relevance. This technique creates a false sense of balance or symmetry between disparate issues, often to downplay or deflect criticism of one's own group while criticizing opponents."
}

system_prompt = "You are a pro-Ukrainian expert who helps crafting messages to facilitate persuasive narratives promoting Ukraine. Please never create two similar narratives for the same prompt - be as diverse as you can."

opening_sentence = "Here is a claim propagated by Russia frequently:"

styles = ['Pessimistic', 'Optimistic', 'Emotional', 'Rational', 'Dry language', 'Metaphorical', 'Amusing', 'Cynical', 'Empathic', 'Detached']

In [ ]:
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
def create_response(base_claim, style, rhet_tech_desc):
    """create a CN based on given rhetorical technique, writing style and a given base claim (also returns additional important data - all in a form of a list)"""
    response = client.chat.completions.create(
      model="llama-3.1-70b-versatile",
      messages=[
          {
              "role": "system",
              "content": str(system_prompt)
          },
          {
              "role": "user",
              "content": str(opening_sentence) + "\n\n\"" + base_claim + "\"\n\nPlease use this rhetorical technique to generate a counter-narrative:\n" + rhet_tech_desc + "\n\n Also, use this writing style: " + style + "\n\nMake it short (up to 35 words), strong and persuasive. Please only provide the counter-narrative without any explanation or opening sentence. Feel free to use a few hashtags that support the narrative."
          }
      ],
      temperature=1.3,
      top_p=1,
      max_tokens=80
    )
    return [response.id, response.choices[0].message.content, len(response.choices[0].message.content.split()), base_claim, rhet_tech_desc.split(":")[0], style, response.model, response.usage.completion_tokens, response.usage.prompt_tokens, response.usage.total_tokens, response.usage.total_time]

In [ ]:
final_df = pd.DataFrame(columns=['response_id', 'response_text', 'num_of_words', 'base_claim', 'rhetorical_technique_name', 'style', 'model_used', 'completion_tokens', 'prompt_tokens', 'total_tokens', 'total time'])

In [ ]:
for c in range(len(russia_bc)):
    for t in rhetorical_techniques:
        for s in styles:
            try:
                row = list(create_response(russia_bc[c], s, rhetorical_techniques[t]))
                new_row_df = pd.DataFrame([row], columns=final_df.columns)
                final_df = pd.concat([final_df, new_row_df], ignore_index=True)
            except:
                time.sleep(5)
                row = list(create_response(russia_bc[c], s, rhetorical_techniques[t]))
                new_row_df = pd.DataFrame([row], columns=final_df.columns)
                final_df = pd.concat([final_df, new_row_df], ignore_index=True)
final_df.insert(0, 'creation_date', datetime.today().strftime('%d/%m/%Y'))

In [ ]:
final_df

In [25]:
final_df.to_csv('cn_dataset_styles.csv', index=False)

In [ ]:
def fix_weird_responses():
    final_df = pd.read_csv('cn_dataset_styles.csv')
    for index, row in final_df[~final_df["response_text"].str.endswith('"')].iterrows():
        row = list(create_response(final_df.loc[index, 'base_claim'], final_df.loc[index, 'style'], rhetorical_techniques[final_df.loc[index, 'rhetorical_technique_name']]))
        row.insert(0, datetime.today().strftime('%d/%m/%Y'))
        final_df.loc[index] = row
    final_df.to_csv('cn_dataset_styles.csv', index=False)

fix_weird_responses()

In [ ]:
def fix_unwanted_length():
    final_df = pd.read_csv('cn_dataset_styles.csv')
    for index, row in final_df.iterrows():
        if row['num_of_words'] > 40 or row['num_of_words'] < 20:
            row = list(create_response(final_df.loc[index, 'base_claim'], final_df.loc[index, 'style'], rhetorical_techniques[final_df.loc[index, 'rhetorical_technique_name']]))
            row.insert(0, datetime.today().strftime('%d/%m/%Y'))
            final_df.loc[index] = row
    final_df.to_csv('cn_dataset_styles.csv', index=False)

fix_unwanted_length()

In [ ]:
def fix_duplicate_responses():
    final_df = pd.read_csv('cn_dataset_styles.csv')
    for index, row in final_df.iterrows():
        is_value_duplicated = final_df[final_df['response_text'] == final_df.loc[index, 'response_text']].index.tolist()
        is_value_duplicated.remove(index)
        if is_value_duplicated:
            row = list(create_response(final_df.loc[index, 'base_claim'], final_df.loc[index, 'style'], rhetorical_techniques[final_df.loc[index, 'rhetorical_technique_name']]))
            row.insert(0, datetime.today().strftime('%d/%m/%Y'))
            final_df.loc[index] = row
    final_df.to_csv('cn_dataset_styles.csv', index=False)

fix_duplicate_responses()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(final_df['response_text'])
cosine_sim_matrix = np.array(cosine_similarity(tfidf_matrix))

for i in range(len(final_df)):
    # let's find all indexes of responses that were generated using the same rhetorical technique and for the same base claim
    other_rows_indexes = [i for i in range(int(i/10) * 10, (int(i/10) + 1) * 10)]
    other_rows_indexes.remove(i)
    for j in other_rows_indexes:
        if cosine_sim_matrix[i, j] >= 0.3:
            print("found one pair:", (i, j))

found one pair: (23, 28)
found one pair: (23, 29)
found one pair: (28, 23)
found one pair: (28, 29)
found one pair: (29, 23)
found one pair: (29, 28)
found one pair: (53, 59)
found one pair: (59, 53)
found one pair: (64, 69)
found one pair: (66, 68)
found one pair: (68, 66)
found one pair: (69, 64)
found one pair: (71, 79)
found one pair: (79, 71)
found one pair: (81, 83)
found one pair: (82, 83)
found one pair: (82, 84)
found one pair: (82, 87)
found one pair: (82, 89)
found one pair: (83, 81)
found one pair: (83, 82)
found one pair: (83, 84)
found one pair: (83, 87)
found one pair: (84, 82)
found one pair: (84, 83)
found one pair: (87, 82)
found one pair: (87, 83)
found one pair: (89, 82)
found one pair: (103, 109)
found one pair: (109, 103)
found one pair: (123, 124)
found one pair: (124, 123)
found one pair: (141, 143)
found one pair: (141, 148)
found one pair: (143, 141)
found one pair: (148, 141)
found one pair: (161, 167)
found one pair: (167, 161)
found one pair: (173, 179)
fou